# External classifier pre-screening on REAL RAVDESS frames

**Question this notebook answers:** which third-party FER classifiers are
*useful evaluators* of our generated videos?

Pre-screen on real (ungenerated) RAVDESS frames across **all three splits**
(train, val, test, plus a merged train+val) for cross-split consistency.
An external is a reliable evaluator only if:
  1. F1_test > ~0.5 (has headroom to differentiate model variants), AND
  2. F1_test ≈ F1_train+val (not a small-test n=96 artefact).

Internal reference: TimeSformer trained in `02_train_encoders_4emotions`
achieves macro F1 = 0.876 on real test (`04b_encoder_ceiling` cell 6).
External F1 below this is the FER↔RAVDESS domain-shift cost.

Use the ranking to pick which externals to keep in `05_external_evaluation`
/ `07a_sadtalker_loss_ablation`. If **no** external clears 0.5, that is
itself a strong methodological finding for the thesis (see closing markdown).


In [1]:
!pip install -q transformers librosa scipy scikit-learn torchaudio

In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys
sys.path.insert(0, "/content")  # so emotion_utils + Wav2Lip imports resolve
sys.path.insert(0, "/content/Wav2Lip")

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix,
)
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModelForImageClassification

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
METADATA = "/content/processed_data/metadata.json"
OUT_DIR = Path("/content/external_screening")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Internal 4-emo encoders trained in 02_train_encoders_4emotions (winners from cell 12).
BEST_AUDIO_PATH = "/content/trained_encoders_4emotions/4emo-w2v2-er-lr3e5"
BEST_VIDEO_PATH = "/content/trained_encoders_4emotions/4emo-tsf-lr3e5-8f"

EMOTIONS = ["happy", "sad", "angry", "disgust"]
EXCLUDE = {0, 1, 5, 7}
REMAP = {2: 0, 3: 1, 4: 2, 6: 3}
NUM_EMO = len(EMOTIONS)
FRAMES_PER_SAMPLE = 8   # how many frames to sample uniformly per video for classification

_NAME_ALIASES = {
    "happy":     ["happy", "happiness", "joy"],
    "sad":       ["sad", "sadness"],
    "angry":     ["angry", "anger"],
    "disgust":   ["disgust", "disgusted"],
}

print(f"Device: {DEVICE}")

Device: cuda


In [3]:
# Candidate external FER classifiers — extend this list as you discover more.
# Each entry: (display_name, hf_repo_id). Models that fail to load (404 / no
# image-classification head / etc.) are reported and skipped — see next cell.
CANDIDATES = [
    # Already used in 05/07 — kept here as the reference floor.
    ("dima806",      "dima806/facial_emotions_image_detection"),
    ("trpakov",      "trpakov/vit-face-expression"),
    # New candidates — popular ViT-based FER alternatives.
    ("motheecreator", "motheecreator/vit-Facial-Expression-Recognition"),
    ("Rajaram1996",   "Rajaram1996/FacialEmoRecog"),
    ("RickyIG",       "RickyIG/emotion_face_image_classification"),
    # SigLIP-2 family if available — newer architecture.
    ("prithivMLmods-siglip2", "prithivMLmods/Facial-Emotion-Detection-SigLIP2"),
    # Add more here as you find them — the loader is fault-tolerant.
]

In [4]:
# Load every candidate; skip ones that fail. The skipped list is shown so you
# can substitute alternates without restarting the kernel.
EXTERNALS = {}
skipped = []
for name, repo in CANDIDATES:
    try:
        proc = AutoImageProcessor.from_pretrained(repo)
        model = AutoModelForImageClassification.from_pretrained(repo).to(DEVICE).eval()
        for p in model.parameters():
            p.requires_grad = False
        id2label = model.config.id2label
        label2id_raw = {v.lower(): k for k, v in id2label.items()}
        target_ids = {}
        for our in EMOTIONS:
            picked = None
            for alias in _NAME_ALIASES.get(our, [our]):
                if alias in label2id_raw:
                    picked = label2id_raw[alias]
                    break
            if picked is None:
                raise KeyError(f"no label for '{our}' in {repo}; got {sorted(label2id_raw)}")
            target_ids[our] = picked
        EXTERNALS[name] = {
            "repo":       repo,
            "proc":       proc,
            "model":      model,
            "id2label":   id2label,
            "target_ids": target_ids,
        }
        print(f"[{name}] {repo}")
        print(f"  labels:        {id2label}")
        print(f"  target→ext id: {target_ids}")
    except Exception as exc:
        skipped.append((name, repo, str(exc).splitlines()[0]))
        print(f"[{name}] SKIPPED ({repo}): {str(exc).splitlines()[0]}")

if skipped:
    print(f"\n{len(skipped)} candidate(s) skipped — replace in CANDIDATES if needed:")
    for n, r, e in skipped:
        print(f"  - {n} ({r}): {e}")
print(f"\n{len(EXTERNALS)} candidate(s) ready for screening.")

# ─────────────────────────────────────────────────────────────
# Internal encoders (in-domain, trained on 4-emo RAVDESS)
# ─────────────────────────────────────────────────────────────
from emotion_utils import (
    DifferentiableVideoPreprocess,
    load_frozen_audio_encoder,
    load_frozen_video_encoder,
)

# Holders separate from EXTERNALS so prediction code can dispatch by type.
INTERNAL_VIDEO = None
INTERNAL_AUDIO = None
try:
    audio_enc, audio_proc = load_frozen_audio_encoder(BEST_AUDIO_PATH, DEVICE)
    INTERNAL_AUDIO = {
        "name":  "internal_audio",
        "repo":  BEST_AUDIO_PATH,
        "model": audio_enc,
        "proc":  audio_proc,
        "head_labels": int(getattr(audio_enc.config, "num_labels", len(EMOTIONS))),
    }
    print(f"[internal_audio] {BEST_AUDIO_PATH} (head={INTERNAL_AUDIO['head_labels']})")
except Exception as exc:
    print(f"[internal_audio] SKIPPED: {exc}")

try:
    video_enc = load_frozen_video_encoder(BEST_VIDEO_PATH, DEVICE)
    video_preprocess = DifferentiableVideoPreprocess(224).to(DEVICE)
    INTERNAL_VIDEO = {
        "name":  "internal_video",
        "repo":  BEST_VIDEO_PATH,
        "model": video_enc,
        "preprocess": video_preprocess,
        "num_frames":  int(getattr(video_enc.config, "num_frames", 8)),
        "head_labels": int(getattr(video_enc.config, "num_labels", len(EMOTIONS))),
    }
    print(f"[internal_video] {BEST_VIDEO_PATH} (T={INTERNAL_VIDEO['num_frames']}, head={INTERNAL_VIDEO['head_labels']})")
except Exception as exc:
    print(f"[internal_video] SKIPPED: {exc}")

print(f"\nReady: {len(EXTERNALS)} external + "
      f"{int(INTERNAL_VIDEO is not None)} internal_video + "
      f"{int(INTERNAL_AUDIO is not None)} internal_audio.")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[dima806] dima806/facial_emotions_image_detection
  labels:        {0: 'sad', 1: 'disgust', 2: 'angry', 3: 'neutral', 4: 'fear', 5: 'surprise', 6: 'happy'}
  target→ext id: {'happy': 6, 'sad': 0, 'angry': 2, 'disgust': 1}
[trpakov] trpakov/vit-face-expression
  labels:        {0: 'angry', 1: 'disgust', 2: 'fear', 3: 'happy', 4: 'neutral', 5: 'sad', 6: 'surprise'}
  target→ext id: {'happy': 3, 'sad': 5, 'angry': 0, 'disgust': 1}
[motheecreator] motheecreator/vit-Facial-Expression-Recognition
  labels:        {0: 'anger', 1: 'disgust', 2: 'fear', 3: 'happy', 4: 'neutral', 5: 'sad', 6: 'surprise'}
  target→ext id: {'happy': 3, 'sad': 5, 'angry': 0, 'disgust': 1}
[Rajaram1996] Rajaram1996/FacialEmoRecog
  labels:        {0: 'anger', 1: 'contempt', 2: 'disgust', 3: 'fear', 4: 'happy', 5: 'neutral', 6: 'sadness', 7: 'surprise'}
  target→ext id: {'happy': 4, 'sad': 6, 'angry': 0, 'disgust': 2}
[RickyIG] RickyIG/emotion_face_image_classification
  labels:        {0: 'anger', 1: 'contempt', 2: 

In [5]:
# Real RAVDESS frames — load ALL splits separately so we can compare
# per-split F1 (train ≈ val ≈ test indicates external is consistent;
# big gap → eval is noisy on small test split).
with open(METADATA) as f:
    meta = json.load(f)

real_samples = {
    split_name: [s for s in meta if s["split"] == split_name and s["emotion_idx"] not in EXCLUDE]
    for split_name in ("train", "val", "test")
}

# train+val merged for "extra signal" on top of test alone.
real_samples["train+val"] = real_samples["train"] + real_samples["val"]

from collections import Counter
for split_name, ss in real_samples.items():
    counts = dict(Counter(EMOTIONS[REMAP[s["emotion_idx"]]] for s in ss))
    print(f"{split_name:10s}: n={len(ss):4d}  per-emotion={counts}")


train     : n= 544  per-emotion={'happy': 136, 'sad': 136, 'angry': 136, 'disgust': 136}
val       : n=  96  per-emotion={'happy': 24, 'sad': 24, 'angry': 24, 'disgust': 24}
test      : n=  96  per-emotion={'happy': 24, 'sad': 24, 'angry': 24, 'disgust': 24}
train+val : n= 640  per-emotion={'happy': 160, 'sad': 160, 'angry': 160, 'disgust': 160}


In [6]:
@torch.no_grad()
def predict_external(sample, ext_name):
    """Per-frame ViT classification → mean softmax → argmax over our 4 emotions."""
    ext = EXTERNALS[ext_name]
    frames = np.load(sample["frames_path"])
    n_total = frames.shape[0]
    if n_total < 1:
        return None
    idx = np.linspace(0, n_total - 1, min(FRAMES_PER_SAMPLE, n_total)).round().astype(int)
    sampled = frames[idx]
    pil_frames = [Image.fromarray(f) for f in sampled]
    inputs = ext["proc"](pil_frames, return_tensors="pt").to(DEVICE)
    logits = ext["model"](**inputs).logits
    probs = F.softmax(logits, dim=-1).cpu()
    mean_probs = probs.mean(dim=0)
    tgt_indices = [ext["target_ids"][e] for e in EMOTIONS]
    return int(mean_probs[tgt_indices].argmax().item())


@torch.no_grad()
def predict_internal_video(sample):
    """TimeSformer-on-clip → single video logit, no per-frame averaging."""
    iv = INTERNAL_VIDEO
    frames = np.load(sample["frames_path"]).astype(np.float32) / 255.0  # (T, H, W, 3)
    n_total = frames.shape[0]
    if n_total < 1:
        return None
    target_t = iv["num_frames"]
    idx = np.linspace(0, n_total - 1, target_t).round().astype(int)
    sampled = frames[idx]                                        # (T, H, W, 3)
    sampled_t = torch.from_numpy(sampled).permute(0, 3, 1, 2).float()
    pv = iv["preprocess"](sampled_t.unsqueeze(0).to(DEVICE))
    out = iv["model"](pixel_values=pv)
    if iv["head_labels"] == NUM_EMO:
        logits = out.logits
    else:
        logits = out.logits[:, [2, 3, 4, 6]]  # WAV2LIP_TO_ENCODER 4-emo slice
    return int(logits.argmax(dim=-1).item())


@torch.no_grad()
def predict_internal_audio(sample):
    """HuBERT/W2V2 categorical FER on the same audio file."""
    ia = INTERNAL_AUDIO
    wav, _ = torchaudio.load(sample["audio_path"])
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav_1d = wav.squeeze(0).numpy()
    sr = getattr(ia["proc"], "sampling_rate", 16000)
    enc = ia["proc"]([wav_1d], sampling_rate=sr, return_tensors="pt",
                     padding="max_length", truncation=True, max_length=int(3.0 * sr))
    kwargs = {"input_values": enc["input_values"].to(DEVICE)}
    if "attention_mask" in enc:
        kwargs["attention_mask"] = enc["attention_mask"].to(DEVICE)
    out = ia["model"](**kwargs).logits
    if ia["head_labels"] == NUM_EMO:
        logits = out
    else:
        logits = out[:, [2, 3, 4, 6]]
    return int(logits.argmax(dim=-1).item())


SPLITS_TO_RUN = ["test", "val", "train", "train+val"]

# Unified encoders dict — name → (kind, predict_fn)
ENCODERS = {}
for name in EXTERNALS:
    ENCODERS[name] = ("external", lambda s, n=name: predict_external(s, n))
if INTERNAL_VIDEO is not None:
    ENCODERS["internal_video"] = ("internal_video", predict_internal_video)
if INTERNAL_AUDIO is not None:
    ENCODERS["internal_audio"] = ("internal_audio", predict_internal_audio)

# results_per_split[split][encoder_name] = DataFrame
results_per_split = {sp: {} for sp in SPLITS_TO_RUN}
for split_name in SPLITS_TO_RUN:
    ss = real_samples[split_name]
    if not ss:
        continue
    for enc_name, (kind, fn) in ENCODERS.items():
        rows = []
        for s in tqdm(ss, desc=f"{split_name} [{enc_name}]"):
            try:
                pred = fn(s)
                if pred is None:
                    continue
                rows.append({
                    "sample_id":    s["sample_id"],
                    "emotion_true": REMAP[s["emotion_idx"]],
                    "ext_pred":     pred,
                })
            except Exception as exc:
                print(f"  {enc_name}/{s['sample_id']}/{split_name} failed: {exc}")
        results_per_split[split_name][enc_name] = pd.DataFrame(rows)


train+val [internal_audio]: 100%|██████████| 640/640 [00:12<00:00, 52.92it/s]


In [7]:
def _per_emotion_block(df):
    if len(df) == 0:
        return None
    y = df["emotion_true"].to_numpy()
    p = df["ext_pred"].to_numpy()
    acc = accuracy_score(y, p)
    f1m = f1_score(y, p, labels=list(range(NUM_EMO)), average="macro", zero_division=0)
    per_f1 = f1_score(y, p, labels=list(range(NUM_EMO)), average=None, zero_division=0)
    cm = confusion_matrix(y, p, labels=list(range(NUM_EMO)))
    return acc, f1m, per_f1, cm


# Long-form: one row per (encoder × split) — base for thesis tables.
long_rows = []
for enc_name in ENCODERS:
    for split_name in SPLITS_TO_RUN:
        df = results_per_split[split_name].get(enc_name, pd.DataFrame())
        block = _per_emotion_block(df)
        row = {
            "encoder":  enc_name,
            "kind":     ENCODERS[enc_name][0],
            "split":    split_name,
            "n":        len(df),
        }
        if block is None:
            row.update({"acc": np.nan, "macro_F1": np.nan,
                         **{f"F1_{e}": np.nan for e in EMOTIONS}})
        else:
            acc, f1m, per_f1, _ = block
            row.update({"acc": acc, "macro_F1": f1m,
                         **{f"F1_{EMOTIONS[i]}": per_f1[i] for i in range(NUM_EMO)}})
        long_rows.append(row)

long_df = pd.DataFrame(long_rows)
long_df.to_csv(OUT_DIR / "screening_long.csv", index=False)

# ─────────────────────────────────────────────────────────────
# Thesis Table 1 — Macro F1 by encoder × split (the headline number)
# ─────────────────────────────────────────────────────────────
macro_pivot = long_df.pivot_table(
    index="encoder", columns="split", values="macro_F1", aggfunc="first")
macro_pivot = macro_pivot[SPLITS_TO_RUN]  # column order
macro_pivot["kind"] = long_df.set_index("encoder")["kind"].to_dict().get
macro_pivot["kind"] = [ENCODERS[e][0] if e in ENCODERS else "" for e in macro_pivot.index]
macro_pivot = macro_pivot.sort_values("test", ascending=False)
macro_pivot = macro_pivot[["kind"] + SPLITS_TO_RUN]

print("=" * 75)
print("TABLE 1 — Macro F1 by encoder × split (real RAVDESS, 4 emotions)")
print("=" * 75)
print(macro_pivot.round(3).to_string())
macro_pivot.round(3).to_csv(OUT_DIR / "table1_macro_f1_by_split.csv")

# ─────────────────────────────────────────────────────────────
# Thesis Table 2 — Per-emotion F1 on TEST split (held-out)
# ─────────────────────────────────────────────────────────────
per_emo_test = long_df[long_df["split"] == "test"][
    ["encoder", "kind", "n", "macro_F1"] + [f"F1_{e}" for e in EMOTIONS]
].copy()
per_emo_test = per_emo_test.sort_values("macro_F1", ascending=False).reset_index(drop=True)
print("\n" + "=" * 75)
print("TABLE 2 — Per-emotion F1 on TEST split (n=96, 24/class)")
print("=" * 75)
print(per_emo_test.round(3).to_string(index=False))
per_emo_test.round(3).to_csv(OUT_DIR / "table2_per_emotion_test.csv", index=False)

# ─────────────────────────────────────────────────────────────
# Thesis Table 3 — Per-emotion F1 on TRAIN+VAL (n=640, more reliable)
# ─────────────────────────────────────────────────────────────
per_emo_tv = long_df[long_df["split"] == "train+val"][
    ["encoder", "kind", "n", "macro_F1"] + [f"F1_{e}" for e in EMOTIONS]
].copy()
per_emo_tv = per_emo_tv.sort_values("macro_F1", ascending=False).reset_index(drop=True)
print("\n" + "=" * 75)
print("TABLE 3 — Per-emotion F1 on TRAIN+VAL split (n=640, 160/class)")
print("=" * 75)
print(per_emo_tv.round(3).to_string(index=False))
per_emo_tv.round(3).to_csv(OUT_DIR / "table3_per_emotion_train_val.csv", index=False)

# ─────────────────────────────────────────────────────────────
# Thesis Table 4 — Test/Train+val gap (consistency check)
# ─────────────────────────────────────────────────────────────
gap_rows = []
for enc in macro_pivot.index:
    f1_t = macro_pivot.loc[enc, "test"]
    f1_tv = macro_pivot.loc[enc, "train+val"]
    gap_rows.append({
        "encoder":     enc,
        "kind":        macro_pivot.loc[enc, "kind"],
        "F1_test":     f1_t,
        "F1_train+val": f1_tv,
        "abs_gap":     abs(f1_t - f1_tv) if pd.notna(f1_t) and pd.notna(f1_tv) else np.nan,
        "consistent":  abs(f1_t - f1_tv) < 0.10 if pd.notna(f1_t) and pd.notna(f1_tv) else False,
    })
gap_df = pd.DataFrame(gap_rows).sort_values("F1_test", ascending=False)
print("\n" + "=" * 75)
print("TABLE 4 — Test ↔ Train+val consistency (small-n test artefact check)")
print("=" * 75)
print(gap_df.round(3).to_string(index=False))
gap_df.round(3).to_csv(OUT_DIR / "table4_split_consistency.csv", index=False)

# ─────────────────────────────────────────────────────────────
# Diagnostic: confusion matrices on TRAIN+VAL (most reliable)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("Confusion matrices on TRAIN+VAL (rows=true, cols=pred)")
print("=" * 75)
for enc_name in macro_pivot.index:
    if enc_name not in ENCODERS:
        continue
    df = results_per_split.get("train+val", {}).get(enc_name, pd.DataFrame())
    if len(df) == 0:
        continue
    cm = confusion_matrix(df["emotion_true"], df["ext_pred"], labels=list(range(NUM_EMO)))
    print(f"\n  {enc_name} (n={len(df)})")
    print(pd.DataFrame(cm, index=EMOTIONS, columns=EMOTIONS).to_string())


TABLE 1 — Macro F1 by encoder × split (real RAVDESS, 4 emotions)
split                     kind   test    val  train  train+val
encoder                                                       
internal_video  internal_video  0.876  0.751  1.000      0.964
internal_audio  internal_audio  0.739  0.728  0.994      0.955
motheecreator         external  0.476  0.660  0.646      0.649
trpakov               external  0.464  0.518  0.528      0.527
Rajaram1996           external  0.462  0.585  0.579      0.581
RickyIG               external  0.432  0.471  0.519      0.514
dima806               external  0.280  0.416  0.385      0.390

TABLE 2 — Per-emotion F1 on TEST split (n=96, 24/class)
       encoder           kind  n  macro_F1  F1_happy  F1_sad  F1_angry  F1_disgust
internal_video internal_video 96     0.876     0.906   0.718     0.923       0.958
internal_audio internal_audio 96     0.739     0.667   0.760     0.721       0.810
 motheecreator       external 96     0.476     0.738   0.133  

## Thesis-friendly outputs

Four CSVs are saved under `OUT_DIR`:

| File | Contents |
|------|----------|
| `table1_macro_f1_by_split.csv` | Encoder × Split macro-F1 — top-level table for Methodology §X |
| `table2_per_emotion_test.csv` | Per-emotion F1 on TEST split — for Results §X (matches H1/H2 evaluation split) |
| `table3_per_emotion_train_val.csv` | Per-emotion F1 on TRAIN+VAL — for Limitations / sample-size discussion |
| `table4_split_consistency.csv` | Test/Train+val gap — flags small-n artefacts |
| `screening_long.csv` | Raw long-form (encoder × split × emotion) — backup for ad-hoc analyses |

## Reading the tables

- **Internal video** (TimeSformer 4-emo) is the in-domain ceiling. Numbers below it
  quantify how much in-domain training matters for RAVDESS-style data.
- **Internal audio** (Wav2Vec2 4-emo) is shown for comparison only — audio is not
  modified by Wav2Lip / SadTalker, so it is not used as an evaluator of generation.
  Its F1 here just confirms the audio modality is a clean source of emotion labels.
- **Externals** (motheecreator / Rajaram1996 / etc.) are the candidates we use as
  independent evaluators in `05` and `07a`. The two retained for downstream use
  are those with `F1_test ≥ 0.4` AND consistent across splits (`abs_gap < 0.10`).

If a candidate's F1 on real test is at noise floor, its judgment of generated
videos should not be quoted as primary evidence in the thesis.